In [0]:
# DIM CATEGORY

from pyspark.sql import functions as F

# Variables from the old notebook do not exist here, so we recreate them
silver = spark.table("workspace.silver.transactions")

In [0]:
silver.groupBy("category").count().orderBy("category").show()

In [0]:
dim_category = (
    silver
    .select("category")
    .distinct()
    .withColumnRenamed("category", "category_name")
    .withColumn("category_key", F.xxhash64("category_name"))
    # Business rule: does the money come in or go out?
    .withColumn(
        "flow_type",
        F.when(F.col("category_name") == "salary", "inflow")
         .when(F.col("category_name").isin("groceries", "health", "leisure", "rent", "transport"), "outflow")
         .otherwise("unknown")
    )
    .select("category_key", "category_name", "flow_type")
)

(
    dim_category.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.dim_category")
)

In [0]:
gold_cat = spark.table("workspace.gold.dim_category")
gold_cat.orderBy("category_name").show(truncate=False)

# rows and distinct keys must match
print("rows:", gold_cat.count(), "| distinct keys:", gold_cat.select("category_key").distinct().count())

# every silver category must exist in the dimension. Expected: 0
silver_cats = silver.select("category").distinct()
missing = silver_cats.join(
    gold_cat, silver_cats["category"] == gold_cat["category_name"], "left_anti"
).count()
print("categories without dimension row:", missing)

# flow_type distribution. Expected: inflow 1, outflow 5, unknown 1
gold_cat.groupBy("flow_type").count().show()